# 08. PLOS Headache Diary Dataset: Separate Model Test

This notebook tests whether the published PLOS ONE smartphone headache diary dataset can be used as a second, separate machine-learning dataset for the MSc project.

The aim is not to replace the existing lifestyle-based migraine model. Instead, this notebook checks whether a clinically richer headache diary dataset can support a separate headache-day prediction model.

## 1. Import required libraries

The notebook uses pandas for data handling and scikit-learn for basic machine-learning models and evaluation metrics.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

## 2. Load the translated working copy

The original supplementary spreadsheet is mainly in Korean. A working English copy was prepared for analysis while keeping the original file unchanged.

In [2]:
DATA_PATH = "data/plos_headache_diary/park_2016_s1_headache_diary_english_working_copy.xlsx"

xls = pd.ExcelFile(DATA_PATH)
xls.sheet_names

['62_patients_translated',
 'headache_diary_1099',
 'total_diary_4579',
 'Translation_notes']

## 3. Read the diary sheet

The sheet `total_diary_4579` contains the full diary records. The first few rows contain multi-level labels, so the notebook reconstructs clearer column names before modelling.

In [3]:
raw = pd.read_excel(DATA_PATH, sheet_name="total_diary_4579", header=None, dtype=object)
raw.shape

(4594, 105)

In [4]:
raw.head(5)

,0,1,2,3,4,5,6,7,8,9,...,95,96,97,98,99,100,101,102,103,104
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,No.,Registration ID,Unique ID,Preventive medication use,Start time,NaN,End time,NaN,Duration,Duration (minutes),...,Exercise status,Exercise intensity,NaN,NaN,Vigorous exercise,Vigorous exercise (minutes),Moderate exercise,Moderate exercise (minutes),Headache-free day,Headache ongoing
2,NaN,NaN,NaN,NaN,NaN,Start time,NaN,End time,NaN,NaN,...,NaN,Vigorous exercise,Moderate exercise,Total,NaN,NaN,NaN,NaN,NaN,NaN
3,911,cmc-0003,M-003,0,201410251830,18,201410252320,23,4 hr 50 min,290,...,N,0,0,0,0 min,0,0 min,0,N,N
4,511,dha-0014,em14,0,201410031430,14,201410032157,21,7 hr 27 min,447,...,N,0,0,0,0 min,0,0 min,0,N,N


## 4. Reconstruct useful column names

The spreadsheet uses a two-row heading structure. This step combines the broad heading and sub-heading into one clearer column name.

In [5]:
def build_column_names(raw_df):
    header_1 = raw_df.iloc[1].fillna("")
    header_2 = raw_df.iloc[2].fillna("")

    columns = []
    last_group = ""
    seen = {}

    for first, second in zip(header_1, header_2):
        first = str(first).strip()
        second = str(second).strip()

        if first and first != "nan":
            last_group = first

        if second:
            name = second
        elif first:
            name = first
        elif last_group:
            name = last_group
        else:
            name = "unnamed"

        if second and last_group and second != last_group:
            name = f"{last_group} - {second}"

        name = name.replace("\n", " ").strip()
        seen[name] = seen.get(name, 0) + 1
        if seen[name] > 1:
            name = f"{name}_{seen[name]}"

        columns.append(name)

    return columns

columns = build_column_names(raw)
df = raw.iloc[3:].reset_index(drop=True)
df.columns = columns

df.head()

,No.,Registration ID,Unique ID,Preventive medication use,Start time,Start time_2,End time,End time_2,Duration,Duration (minutes),...,Exercise status,Exercise intensity - Vigorous exercise,Exercise intensity - Moderate exercise,Exercise intensity - Total,Vigorous exercise,Vigorous exercise (minutes),Moderate exercise,Moderate exercise (minutes),Headache-free day,Headache ongoing
0,911,cmc-0003,M-003,0,201410251830,18,201410252320,23,4 hr 50 min,290,...,N,0,0,0,0 min,0,0 min,0,N,N
1,511,dha-0014,em14,0,201410031430,14,201410032157,21,7 hr 27 min,447,...,N,0,0,0,0 min,0,0 min,0,N,N
2,231,cmc-0022,M-011,1,201409201404,14,201409201704,17,3 hr 0 min,180,...,Y,0,1,1,0 hr 0 min,0,0 hr 40 min,40,N,N
3,280,cmc-0022,M-011,1,201409220857,08,201409221257,12,4 hr 0 min,240,...,Y,0,1,1,0 hr 0 min,0,1 hr 0 min,60,N,N
4,307,cmc-0022,M-011,1,201409231800,18,201409232000,20,2 hr 0 min,120,...,Y,0,1,1,0 hr 0 min,0,1 hr 0 min,60,N,N


## 5. Inspect the key outcome fields

This dataset contains several possible outcome fields. For a clean second model, the most defensible task is predicting whether a diary record is a headache day or a headache-free day.

The field `Headache-free day` is used to define the binary target:

- `0` = headache-free day
- `1` = headache day

In [6]:
key_columns = [
    "Headache-free day",
    "Associated symptoms - Migraine yes/no",
    "Severity",
    "Severity - Intensity",
    "Index/score",
]

for column in key_columns:
    if column in df.columns:
        print(f"\n{column}")
        print(df[column].value_counts(dropna=False).head(20))


Headache-free day
Headache-free day
Y      3491
N      1002
NaN      98
Name: count, dtype: int64

Associated symptoms - Migraine yes/no
Associated symptoms - Migraine yes/no
NaN    3493
0       762
1       336
Name: count, dtype: int64

Severity
Severity
NaN         3525
Mild         547
Moderate     402
심함           117
Name: count, dtype: int64

Severity - Intensity
Severity - Intensity
NaN    3525
0       547
1       519
Name: count, dtype: int64

Index/score
Index/score
NaN    3598
3       179
5       165
4       150
2       138
6       100
1        76
7        75
8        60
9        40
10       10
Name: count, dtype: int64


## 6. Convert coded values

The diary uses simple coded values such as `Y` and `N`. These are converted into numeric values so that the machine-learning models can process them.

In [7]:
clean_df = df.copy()

value_map = {
    "Y": 1,
    "N": 0,
    "Yes": 1,
    "No": 0,
    "Mild": 0,
    "Moderate": 1,
    "Severe": 2,
    "심함": 2,
}

for column in clean_df.columns:
    clean_df[column] = clean_df[column].replace(value_map)

## 7. Define a leakage-aware modelling task

A migraine vs non-migraine model can become artificially strong if symptom fields used to define migraine are included as predictors. To avoid this, this notebook uses a broader and more appropriate task:

**Predict headache day vs headache-free day using trigger and lifestyle diary fields.**

This is closer to a self-management use case because the model uses potential trigger information rather than diagnostic criteria. Post-headache improvement factors such as sleep/rest/massage after symptoms are excluded from the feature set because they may introduce data leakage.

In [8]:
target_column = "Headache-free day"

model_df = clean_df[clean_df[target_column].notna()].copy()

# Headache-free day: 1 means headache-free, 0 means headache occurred.
# For modelling, target = 1 means headache day.
y = (pd.to_numeric(model_df[target_column], errors="coerce") == 0).astype(int)

feature_keywords = [
    "Preventive medication use",
    "Trigger factors",
    "Internal factors",
    "External factors",
    "Other",
    "Exercise status",
    "Exercise intensity",
    "Vigorous exercise",
    "Moderate exercise",
]

feature_columns = [
    column for column in model_df.columns
    if any(keyword in column for keyword in feature_keywords)
    and "Other trigger factors" not in column
]

X = model_df[feature_columns].apply(pd.to_numeric, errors="coerce")

# Keep features with enough usable values and actual variation.
X = X.loc[:, X.notna().sum() >= len(X) * 0.5]
X = X.loc[:, X.nunique(dropna=True) > 1]

print("Rows:", X.shape[0])
print("Features:", X.shape[1])
print("Headache days:", int(y.sum()))
print("Headache-free days:", int((y == 0).sum()))

X.head()

Rows: 4493
Features: 31
Headache days: 1002
Headache-free days: 3491


,Preventive medication use,Trigger factors - Number of triggers,Internal factors - Stress,Internal factors - Excess sleep,Internal factors - Sleep deprivation,Internal factors - Exercise,Internal factors - No exercise,Internal factors - Physical fatigue,Internal factors - Menstrual phase,Internal factors - Ovulation phase,...,Other - Excess caffeine drinks,Other - Excess smoking,Other - Cheese/chocolate,Other - Travel,Other_2,Other - Total,Exercise status,Exercise intensity - Vigorous exercise,Exercise intensity - Moderate exercise,Exercise intensity - Total
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,1
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,1
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,1


## 8. Split the data

The dataset is split into training and testing sets. Stratification is used because headache-free days are more common than headache days.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train balance:")
print(y_train.value_counts(normalize=True))
print("y_test balance:")
print(y_test.value_counts(normalize=True))

X_train: (3594, 31)
X_test: (899, 31)
y_train balance:
Headache-free day
0    0.77685
1    0.22315
Name: proportion, dtype: float64
y_test balance:
Headache-free day
0    0.777531
1    0.222469
Name: proportion, dtype: float64


## 9. Train baseline models

Three standard models are tested:

- Logistic Regression
- Random Forest
- Gradient Boosting

Missing values are handled using median imputation. Logistic Regression also uses feature scaling.

In [10]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
        )),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=42)),
    ]),
}

results = []
fitted_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[model_name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    })

results_df = pd.DataFrame(results).sort_values(
    by=["ROC-AUC", "F1-score"],
    ascending=False,
)

results_df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
1,Random Forest,0.828699,0.618557,0.600,0.609137,0.790783
2,Gradient Boosting,0.838710,0.723577,0.445,0.551084,0.789002
0,Logistic Regression,0.819800,0.592233,0.610,0.600985,0.777307


## 10. Evaluate the best model

The best model is selected using ROC-AUC first and F1-score second. The confusion matrix and classification report show how well the model identifies headache days.

In [11]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

best_predictions = best_model.predict(X_test)

print("Best model:", best_model_name)
print("Confusion matrix:")
print(confusion_matrix(y_test, best_predictions))
print()
print("Classification report:")
print(classification_report(y_test, best_predictions, target_names=["Headache-free day", "Headache day"]))


Best model: Random Forest
Confusion matrix:
[[625  74]
 [ 80 120]]

Classification report:
                   precision    recall  f1-score   support

Headache-free day       0.89      0.89      0.89       699
     Headache day       0.62      0.60      0.61       200

         accuracy                           0.83       899
        macro avg       0.75      0.75      0.75       899
     weighted avg       0.83      0.83      0.83       899



## 11. Feature importance for tree-based models

If the best model is tree-based, feature importance can help identify which diary variables contributed most to prediction. This should be interpreted cautiously because this is still exploratory analysis.

In [12]:
if best_model_name in ["Random Forest", "Gradient Boosting"]:
    importances = best_model.named_steps["model"].feature_importances_
    feature_importance_df = pd.DataFrame({
        "Feature": X.columns,
        "Importance": importances,
    }).sort_values("Importance", ascending=False)

    display(feature_importance_df.head(15))
else:
    print("Feature importance is not shown for this model type in this notebook.")

,Feature,Importance
1,Trigger factors - Number of triggers,0.236099
11,Internal factors - Total,0.130673
25,Other_2,0.072204
6,Internal factors - No exercise,0.055406
26,Other - Total,0.049808
2,Internal factors - Stress,0.048302
0,Preventive medication use,0.044538
4,Internal factors - Sleep deprivation,0.032276
21,Other - Excess caffeine drinks,0.030776
8,Internal factors - Menstrual phase,0.029018


## 12. Benchmark against the published paper

The original PLOS ONE paper is used as the reference benchmark for dataset context and clinically relevant trigger variables:

Park JW et al. *Analysis of Trigger Factors in Episodic Migraineurs Using a Smartphone Headache Diary Application*. PLOS ONE. 2016. DOI: 10.1371/journal.pone.0149577.

The paper reports a smartphone headache diary dataset with 62 participants, 4,579 diary records, 1,099 headache diary records, and 336 migraine records. It focuses on headache/migraine trigger analysis rather than an embedded web prediction prototype, so the benchmark is mainly methodological and dataset-based rather than a direct model-to-model metric comparison.

For this dissertation prototype, the dataset is repurposed into a leakage-aware machine-learning task: predicting headache day vs headache-free day from diary trigger variables. This provides a clinically richer second model while keeping it separate from the original lifestyle-based migraine dataset.

## 13. Brief conclusion

This PLOS headache diary dataset appears suitable as a **separate second dataset** for the project. It should not be merged directly with the current Kaggle-style lifestyle dataset because the structure, source, and target variables are different.

A defensible modelling task is headache-day prediction using trigger and lifestyle diary variables. This aligns better with headache self-management because it focuses on diary-based risk factors rather than directly predicting from diagnostic symptom criteria.

The refined model includes internal triggers, environmental triggers, dietary/other triggers, medication use and exercise variables, while excluding likely post-headache improvement factors to reduce leakage risk. The next step is to use these variables in the web input form and discuss the final feature set with the supervisor.